# 0. Introdução

In [1]:
import io
import sqlite3
import pandas as pd
import os

if os.path.exists('leitos.db'):
    os.remove('leitos.db')

df = pd.read_csv('Leitos_2024.csv', encoding='utf-8', encoding_errors='ignore')
conn = sqlite3.connect('leitos.db')
cur = conn.cursor()
cur.execute("PRAGMA journal_mode=OFF;")

df['COMP'] = df['COMP'].astype(str)
df = df.loc[df['COMP'] == "202411"]

cur.execute('''
CREATE TABLE IF NOT EXISTS Estabelecimento (
    CNES VARCHAR(8) PRIMARY KEY, 
    CEP VARCHAR(9),
    Codigo_Tipo_Unidade VARCHAR(3),
    Natureza_Juridica VARCHAR(5),
    Competencia VARCHAR(7),
    Motivo_Desabilitacao VARCHAR(20),
    Nome VARCHAR(100),
    Razao_Social VARCHAR(20),
    Nome_Logradouro VARCHAR(100),
    Numero_Endereco VARCHAR(10),
    Complemento VARCHAR(10),
    Telefone VARCHAR(15),
    Email VARCHAR(20),
    FOREIGN KEY(CEP) REFERENCES Endereco(CEP),
    FOREIGN KEY(Codigo_Tipo_Unidade) REFERENCES Unidade(Codigo_Tipo_Unidade),
    FOREIGN KEY(Natureza_Juridica) REFERENCES Natureza_Juridica(Natureza_Juridica)
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS Endereco (
    CEP VARCHAR(9) PRIMARY KEY,
    Regiao VARCHAR(13),
    UF VARCHAR(3),
    Municipio VARCHAR(20),
    Nome_Bairro VARCHAR(25)
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS Unidade (
    Codigo_Tipo_Unidade VARCHAR(3) PRIMARY KEY,
    Tipo_Gestao VARCHAR(2),
    Descricao_Tipo_Unidade VARCHAR(20)
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS Natureza_Juridica (
    Natureza_Juridica VARCHAR(5) PRIMARY KEY,
    Descricao_Natureza_Juridica VARCHAR(20)
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS Leito (
    Indice INTEGER PRIMARY KEY,
    Tipo VARCHAR(50)
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS Estabelecimento_Leito (
    CNES VARCHAR(8),
    Indice INTEGER,
    Quantidade INTEGER,
    FOREIGN KEY(CNES) REFERENCES Estabelecimento(CNES),
    FOREIGN KEY(Indice) REFERENCES Leito(Indice)
)
''')

estabelecimento_data = []
for index, row in df.iterrows():
    estabelecimento_data.append((
        row['CNES'],
        row['CO_CEP'],
        row['CO_TIPO_UNIDADE'],
        row['NATUREZA_JURIDICA'],
        row['COMP'],
        row['MOTIVO_DESABILITACAO'],
        row['NOME_ESTABELECIMENTO'],
        row['RAZAO_SOCIAL'],
        row['NO_LOGRADOURO'],
        row['NU_ENDERECO'],
        row['NO_COMPLEMENTO'],
        row['NU_TELEFONE'],
        row['NO_EMAIL']
    ))

cur.executemany('''
    INSERT INTO Estabelecimento (
        CNES, 
        CEP, 
        Codigo_Tipo_Unidade, 
        Natureza_Juridica, 
        Competencia, 
        Motivo_Desabilitacao, 
        Nome, 
        Razao_Social, 
        Nome_Logradouro, 
        Numero_Endereco, 
        Complemento, 
        Telefone, 
        Email
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''', estabelecimento_data)

endereco_data = []
for index, row in df.iterrows():
    endereco_data.append((
        row['CO_CEP'],
        row['REGIAO'],
        row['UF'],
        row['MUNICIPIO'],
        row['NO_BAIRRO']
    ))

cur.executemany('''
    INSERT OR REPLACE INTO Endereco (
        CEP, 
        Regiao, 
        UF, 
        Municipio, 
        Nome_Bairro
    ) VALUES (?, ?, ?, ?, ?)
''', endereco_data)

tipo_unidade_data = []
for index, row in df.iterrows():
    tipo_unidade_data.append((
        row['CO_TIPO_UNIDADE'],
        row['TP_GESTAO'],
        row['DS_TIPO_UNIDADE']
    ))

cur.executemany('''
    INSERT OR REPLACE INTO Unidade (
        Codigo_Tipo_Unidade, 
        Tipo_Gestao, 
        Descricao_Tipo_Unidade
    ) VALUES (?, ?, ?)
''', tipo_unidade_data)

natureza_juridica_data = []
for index, row in df.iterrows():
    natureza_juridica_data.append((
        row['NATUREZA_JURIDICA'],
        row['DESC_NATUREZA_JURIDICA']
    ))

cur.executemany('''
    INSERT OR REPLACE INTO Natureza_Juridica (
        Natureza_Juridica, 
        Descricao_Natureza_Juridica
    ) VALUES (?, ?)
''', natureza_juridica_data)

tipo_leitos = [
    (1, 'Número de Leitos Existentes'),
    (2, 'Número de Leitos SUS'),
    (3, 'Número de Leitos de UTI Existentes'),
    (4, 'Número de Leitos de UTI SUS'),
    (5, 'Número de Leitos de UTI Adulto Existentes'),
    (6, 'Número de Leitos de UTI Adulto SUS'),
    (7, 'Número de Leitos de UTI Pediátrico Existentes'),
    (8, 'Número de Leitos de UTI Pediátrico SUS'),
    (9, 'Número de Leitos de UTI Neonatal Existentes'),
    (10, 'Número de Leitos de UTI Neonatal SUS'),
    (11, 'Número de Leitos de UTI para Queimados Existentes'),
    (12, 'Número de Leitos de UTI para Queimados SUS'),
    (13, 'Número de Leitos de UTI Coronariana Existentes'),
    (14, 'Número de Leitos de UTI Coronariana SUS')
]

cur.executemany('''
    INSERT INTO Leito (Indice, Tipo) 
    VALUES (?, ?)
''', tipo_leitos)

tipo_leitos_quantidade = {
    1: 'LEITOS_EXISTENTES',
    2: 'LEITOS_SUS',
    3: 'UTI_TOTAL_EXIST',
    4: 'UTI_TOTAL_SUS',
    5: 'UTI_ADULTO_EXIST',
    6: 'UTI_ADULTO_SUS',
    7: 'UTI_PEDIATRICO_EXIST',
    8: 'UTI_PEDIATRICO_SUS',
    9: 'UTI_NEONATAL_EXIST',
    10: 'UTI_NEONATAL_SUS',
    11: 'UTI_QUEIMADO_EXIST',
    12: 'UTI_QUEIMADO_SUS',
    13: 'UTI_CORONARIANA_EXIST',
    14: 'UTI_CORONARIANA_SUS'
}

estabelecimento_tipo_leito_data = []
for index, row in df.iterrows():
    for tipo_leito_id, coluna_nome in tipo_leitos_quantidade.items():
        estabelecimento_tipo_leito_data.append((
            row['CNES'],
            tipo_leito_id,
            row[coluna_nome]
        ))

cur.executemany('''
    INSERT INTO Estabelecimento_Leito (CNES, Indice, Quantidade)
    VALUES (?, ?, ?)
''', estabelecimento_tipo_leito_data)

conn.commit()

# 1. Título

- Hospitais e Leitos

# 2. Membros (nome e número de matrícula)

- Luis Felipe Pimenta Marcos 2023087907

- Leonardo Cesar Cota de Castro 2023087923

- Bernardo Borges machado 2023111115

# 3. Descrição dos dados (qual a URL? qual o domínio? como os dados foram processados?)

- Os dados foram retirados do site https://dados.gov.br/dados/conjuntos-dados/hospitais-e-leitos, de domínio dados.gov.br. Além disso, foram importados no formato CSV e transformados em um dataframe usando a biblioteca pandas, disponível no python. Logo em seguida foi feita a conexão desse data frame usando o SGDB sqlite, e com ele foram geradas as tabelas do banco de dados propriamente dito. Após isso as consultas foram realizadas usando a linguagem SQL normalmente.

# 4. Diagrama ER

- Imagem anexada ao diretório.

# 5. Diagrama relacional

#### Tabela Estabelecimento:
- CNES (PK)
- CEP (FK)
- Codigo_Tipo_Unidade (FK)
- Natureza_Juridica (FK)
- Competencia
- Motivo_Desabilitacao
- Motivo_Desabilitacao
- Razao_Social
- Nome_Logradouro
- Numero_Endereco
- Complemento
- Telefone
- Email

#### Tabela Endereco:
- CEP (PK)
- Regiao
- UF
- Municipio
- Nome_Bairro

#### Tabela Unidade:
- Codigo_Tipo_Unidade (PK)
- Tipo_Gestao
- Descricao_Tipo_Unidade

#### Tabela Natureza_Juridica:
- Natureza_Juridica (PK)
- Descricao_Natureza_Juridica

#### Tabela Leito:
- Indice (PK)
- Tipo

#### Tabela Estabelecimento_Leito:
- CNES (FK)
- Indice (FK)
- Quantidade

# 6. Consultas

## 6.1 Duas consultas envolvendo seleção e projeção

### 6.1.1 Consulta 1

- Seleciona os números de natureza jurídica dos hospitais filantrópicos.

In [ ]:
query = """

SELECT Natureza_Juridica
FROM Natureza_Juridica
WHERE Descricao_Natureza_Juridica = 'HOSPITAL_FILANTRPICO'

"""

df = pd.read_sql_query(query, conn)
df

### 6.1.2 Consulta 2

- Pega os endereços dos estabelecimentos de Juíz de Fora.

In [ ]:
query = """

SELECT CNES, CEP, Nome_Logradouro, Numero_Endereco, Complemento
FROM Estabelecimento
NATURAL JOIN Endereco
WHERE Municipio = 'JUIZ DE FORA'

"""

df = pd.read_sql_query(query, conn)
df

## 6.2 Três consultas envolvendo junção de duas relações

### 6.2.1 Consulta 3

- Seleciona os contatos dos estabelecimentos do bairro Funcionários.

In [ ]:
query = """

SELECT CNES, Nome, Telefone, Email
FROM Estabelecimento
NATURAL JOIN Endereco
WHERE Municipio = 'BELO HORIZONTE' AND Nome_Bairro = 'FUNCIONARIOS'

"""

df = pd.read_sql_query(query, conn)
df

### 6.2.2 Consulta 4

- Pega os municípios que possuem apenas estabelecimentos de natureza jurídica privada.

In [ ]:
query = """

SELECT DISTINCT UF, Municipio
FROM Endereco
WHERE NOT EXISTS (
    SELECT *
    FROM Estabelecimento
    NATURAL JOIN Natureza_Juridica
    WHERE Estabelecimento.CEP = Endereco.CEP
    AND Descricao_Natureza_Juridica != 'HOSPITAL_PRIVADO' )

"""

df = pd.read_sql_query(query, conn)
df

### 6.2.3 Consulta 5

- Ordena os dez municípios com a maior quantidade de estabelecimentos de saúde.

In [ ]:
query = """

SELECT Municipio, COUNT(CNES) as Total
FROM Estabelecimento
NATURAL JOIN Endereco
GROUP BY Municipio
ORDER BY Total DESC
LIMIT 10

"""

df = pd.read_sql_query(query, conn)
df

## 6.3 Três consultas envolvendo junção de três ou mais relações

### 6.3.1 Consulta 6

- Ordena os dez municípios com a maior quantidade de leitos.

In [ ]:
query = """

SELECT Municipio, SUM(Quantidade) AS Total
FROM Estabelecimento
NATURAL JOIN Endereco
NATURAL JOIN Estabelecimento_Leito
NATURAL JOIN Leito
WHERE Tipo = 'Número de Leitos Existentes'
GROUP BY Municipio
ORDER BY Total DESC
LIMIT 10

"""

df = pd.read_sql_query(query, conn)
df

### 6.3.2 Consulta 7

- Pega o total de leitos por estado.

In [ ]:
query = """

SELECT UF, SUM(Quantidade) AS Total
FROM Estabelecimento
NATURAL JOIN Estabelecimento_Leito
NATURAL JOIN Leito
NATURAL JOIN Endereco
WHERE Tipo = 'Número de Leitos Existentes'
GROUP BY UF
ORDER BY Total

"""

df = pd.read_sql_query(query, conn)
df

### 6.3.3 Consulta 8

- Seleciona a quantidade de leitos de cada tipo, dos hospitais de Belo Horizonte.

In [ ]:
query = """

SELECT Nome, Tipo, Quantidade
FROM Estabelecimento
NATURAL JOIN Estabelecimento_Leito
NATURAL JOIN Leito
NATURAL JOIN Endereco
WHERE Municipio = 'BELO HORIZONTE'
AND Tipo IN ( 'Número de Leitos de UTI Adulto Existentes',
              'Número de Leitos de UTI Adulto SUS',
              'Número de Leitos de UTI Pediátrico Existentes',
              'Número de Leitos de UTI Pediátrico SUS',
              'Número de Leitos de UTI Neonatal Existentes',
              'Número de Leitos de UTI Neonatal SUS',
              'Número de Leitos de UTI Coronariana Existentes',
              'Número de Leitos de UTI Coronariana SUS' )
AND quantidade > 0
ORDER BY Quantidade DESC

"""

df = pd.read_sql_query(query, conn)
df

## 6.4 Duas consultas envolvendo agregação sobre junção de duas ou mais relações

### 6.4.1 Consulta 9

- Mostra a quantidade de hospitais por bairro, levando em conta a sua natureza jurídica.

In [ ]:
query = """

SELECT Nome_Bairro, Descricao_Natureza_Juridica, COUNT(*) AS Total
FROM Estabelecimento
NATURAL JOIN Endereco
NATURAL JOIN Natureza_Juridica
WHERE Municipio = 'BELO HORIZONTE'
GROUP BY Nome_Bairro, Descricao_Natureza_Juridica

"""

df = pd.read_sql_query(query, conn)
df

### 6.4.2 Consulta 10

- Revela a média de leitos SUS no Nordeste, em relação aos estados.

In [ ]:
query = """

SELECT Tipo, ROUND(AVG(Soma), 1) AS Media
FROM (
    SELECT Tipo, SUM(Quantidade) AS Soma
    FROM Estabelecimento
    NATURAL JOIN Estabelecimento_Leito
    NATURAL JOIN Leito
    NATURAL JOIN Endereco
    WHERE Regiao = 'NORDESTE'
    AND Tipo IN ( 'Número de Leitos SUS',
                  'Número de Leitos de UTI SUS',
                  'Número de Leitos de UTI Adulto SUS',
                  'Número de Leitos de UTI Pediátrico SUS',
                  'Número de Leitos de UTI Neonatal SUS',
                  'Número de Leitos de UTI para Queimados SUS',
                  'Número de Leitos de UTI Coronariana SUS' )
    GROUP BY UF, Tipo )
GROUP BY Tipo

"""

df = pd.read_sql_query(query, conn)
df

# 7. Autoavaliação dos membros

- Todos os membros do grupo participaram de todas as etapas do trabalho. Na escolha do banco de dados, a escolha foi conjunta depois de trazermos diferentes opções. A construção da proposta contou com a contribuição de todos para alinhar os objetivos do grupo. A conexão do dataframe, criação de tabelas e elaboração dos diagramas relacional e ER foram realizadas de forma colaborativa, com ajustes coletivos. Na parte de consultas, todos contribuíram para criar e otimizar comandos SQL, e cada um foi montando diferentes e ajudando nas revisões. De maneira geral, o trabalho todo contou com a contribuição geral dos membros.